# cc-finance-1.3 — Stage 2 SFT: do inferred thoughts help next-action prediction?

Snorkel-finance, Qwen3-4B-Instruct-2507. Plots the offline SFT training curves per **variant × regime**, comparing the **whole (thought+action) span** against the **action sub-span** (`<tool_call>…`/`Final Answer:` tokens only, isolated by `action_start_token`).

**Headline metric:** `eval_actiononly_ppl` / `eval_actiononly_accuracy` — the whole-span ppl is inflated by verbose thoughts and is misleading (on airline it wrongly ranked `expert_thoughts` worst; the action-subspan flipped it). Variants: `actions_only` (expert action) · `expert_thoughts` (oracle) · `thoughts_policy`/`thoughts_base` (Act-PRM, best + last). Regimes: hide-obs (solid) vs full-context (dashed).

In [ ]:
import cc_finance_lib as L
runs = L.load_sft_runs()
print('SFT runs found:')
for (v, r) in sorted(runs, key=lambda k: (list(L.VARIANTS).index(k[0]), k[1])):
    print(f'  {v:22s} [{r:4s}]  train_rows={len(runs[(v,r)]["train"])}  eval_pts={len(runs[(v,r)]["eval"])}')

## Eval curves (25 held-out act_prm_eval tasks) — whole-span vs action-subspan

In [ ]:
L.plot_sft_grid(runs, 'eval', 'figs_finance/sft_eval_curves.png');

## Train curves (per-step, teacher-forced) — now symmetric with eval (action-subspan added)

In [ ]:
L.plot_sft_grid(runs, 'train', 'figs_finance/sft_train_curves.png');

## Best action-subspan per run (the ranking that matters)

In [ ]:
import pandas as pd
df = pd.DataFrame(L.best_table(runs)).sort_values(['regime', 'best_actiononly_ppl'])
df

## Reading the results
- **Compare within a regime** (hide vs hide, full vs full): does adding a thought before the action lower `best_actiononly_ppl` / raise accuracy vs `actions_only`?
- **whole-span vs action-subspan:** for thought variants, whole-span ppl is inflated by the thought tokens; the action-subspan isolates action fit. If action-subspan improves while whole-span worsens, the thought *helps the action* (the airline flip).
- **best vs last / policy vs base:** which EM snapshot & scorer yields the most useful Act-PRM thoughts.

_Preliminary (partial run): `expert_thoughts` [hide] action-ppl 2.65 < `actions_only` [hide] 2.84 while its whole-span 3.05 > 2.84 — the flip reproduces. Act-PRM (`thoughts_policy/base`) rows populate after Stage 1.5 relabel; re-run this cell to refresh._